# AutoScout24 secondary details scraper

This notebook reads the primary Audi Q4 e-tron CSV, visits each `listing_url`, parses the detail page `__NEXT_DATA__` payload, and saves a merged `_secondary.csv` file.

Note: AutoScout24 detail pages expose `noOfPreviousOwners`, not a dedicated `first owner` boolean, so this notebook stores `previous_owner_count` instead of inventing a field that is not explicitly present.

In [4]:
from __future__ import annotations

import json
import re
import time
import urllib.error
import urllib.request
from http.cookiejar import CookieJar
from pathlib import Path
from datetime import datetime
import pandas as pd

FILE_GLOB = "scrape_audi_q4_*.csv"

# Automatically resolve to the most recently modified primary CSV so that the
# secondary scraper always uses the file from the day the main scrape ran,
# regardless of which day this notebook is executed.
_primary_csvs = sorted(
    [p for p in Path(".").glob(FILE_GLOB) if not p.stem.endswith("_secondary")],
    key=lambda p: p.stat().st_mtime,
    reverse=True,
)
INPUT_CSV_PATH = _primary_csvs[0] if _primary_csvs else None

REQUEST_TIMEOUT_SECONDS = 30
REQUEST_DELAY_SECONDS = 0.6
MAX_RETRIES = 3
MAX_LISTINGS = None  # Set to 5 while testing the notebook.

USER_AGENT = (
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
    "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
)

NEXT_DATA_PATTERN = re.compile(
    r'<script id="__NEXT_DATA__" type="application/json">(.*?)</script>',
    flags=re.DOTALL,
)

SECONDARY_COLUMNS = [
    "listing_url",
    "warranty_exists",
    "warranty_text",
    "has_full_service_history",
    "had_accident",
    "damage_conditions",
    "previous_owner_count",
    "body_type",
    "door_count",
    "seat_count",
    "exterior_color",
    "paint_type",
    "interior_color",
    "upholstery_material",
    "battery_ownership",
    "battery_charging_time",
    "electric_range",
    "electric_range_city",
    "secondary_scrape_status",
    "secondary_scrape_error",
]

AttributeError: partially initialized module 'pandas' from '/opt/miniconda3/lib/python3.13/site-packages/pandas/__init__.py' has no attribute '_pandas_datetime_CAPI' (most likely due to a circular import)

In [ ]:
def clean_text(value):
    if value is None:
        return None
    text = re.sub(r"\s+", " ", str(value)).strip()
    return text or None


def format_nested_value(value):
    if value is None:
        return None

    if isinstance(value, dict):
        for key in ("formatted", "label", "value", "description"):
            candidate = clean_text(value.get(key))
            if candidate is not None:
                return candidate

        raw_value = value.get("raw")
        if raw_value is not None:
            return clean_text(raw_value)

        compact = {key: item for key, item in value.items() if item not in (None, "", [], {})}
        return json.dumps(compact, ensure_ascii=False) if compact else None

    if isinstance(value, list):
        parts = [format_nested_value(item) for item in value]
        parts = [part for part in parts if part not in (None, "")]
        return "; ".join(str(part) for part in parts) if parts else None

    if isinstance(value, bool):
        return value

    if isinstance(value, (int, float)):
        return value

    return clean_text(value)


def build_http_opener():
    cookie_jar = CookieJar()
    opener = urllib.request.build_opener(urllib.request.HTTPCookieProcessor(cookie_jar))
    opener.addheaders = [
        ("User-Agent", USER_AGENT),
        ("Accept-Language", "en-GB,en;q=0.9"),
    ]
    return opener


def find_input_csv_path(input_csv_path=None):
    if input_csv_path:
        return Path(input_csv_path)

    candidates = [
        path for path in Path.cwd().glob(FILE_GLOB)
        if path.is_file() and not path.stem.endswith("_secondary")
    ]
    if not candidates:
        raise FileNotFoundError(
            "No primary CSV was found. Set INPUT_CSV_PATH or add a scrape_audi_q4_*.csv file."
        )

    return max(candidates, key=lambda path: path.stat().st_mtime)


def build_output_csv_path(input_csv_path):
    input_csv_path = Path(input_csv_path)
    return input_csv_path.with_name(f"{input_csv_path.stem}_secondary.csv")


def fetch_html(url, opener):
    last_error = None

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            with opener.open(url, timeout=REQUEST_TIMEOUT_SECONDS) as response:
                return response.read().decode("utf-8", errors="ignore")
        except (urllib.error.HTTPError, urllib.error.URLError, TimeoutError) as exc:
            last_error = exc
            if attempt < MAX_RETRIES:
                time.sleep(REQUEST_DELAY_SECONDS * attempt)

    raise RuntimeError(f"Failed to fetch {url}: {last_error}")


def extract_next_data(html):
    match = NEXT_DATA_PATTERN.search(html)
    if not match:
        raise ValueError("Could not find __NEXT_DATA__ on the detail page.")
    return json.loads(match.group(1))


def extract_secondary_fields(listing_details):
    vehicle = listing_details.get("vehicle") or {}

    return {
        "warranty_exists": listing_details.get("warrantyExists"),
        "warranty_text": format_nested_value(listing_details.get("warranty")),
        "has_full_service_history": vehicle.get("hasFullServiceHistory"),
        "had_accident": vehicle.get("hadAccident"),
        "damage_conditions": format_nested_value(vehicle.get("damageConditions")),
        "previous_owner_count": vehicle.get("noOfPreviousOwners"),
        "body_type": format_nested_value(vehicle.get("bodyType")),
        "door_count": vehicle.get("numberOfDoors"),
        "seat_count": vehicle.get("numberOfSeats"),
        "exterior_color": format_nested_value(vehicle.get("bodyColor")),
        "paint_type": format_nested_value(vehicle.get("paintType")),
        "interior_color": format_nested_value(vehicle.get("upholsteryColor")),
        "upholstery_material": format_nested_value(vehicle.get("upholstery")),
        "battery_ownership": format_nested_value(vehicle.get("batteryOwnershipType")),
        "battery_charging_time": format_nested_value(vehicle.get("batteryChargingTime")),
        "electric_range": format_nested_value(vehicle.get("electricRangeWithFallback")),
        "electric_range_city": format_nested_value(vehicle.get("electricRangeCity")),
    }

In [ ]:
def fetch_secondary_details(listing_url, opener):
    record = {column: None for column in SECONDARY_COLUMNS}
    record["listing_url"] = clean_text(listing_url)

    try:
        html = fetch_html(listing_url, opener)
        next_data = extract_next_data(html)
        listing_details = next_data["props"]["pageProps"]["listingDetails"]
        record.update(extract_secondary_fields(listing_details))
        record["secondary_scrape_status"] = "ok"
    except Exception as exc:
        record["secondary_scrape_status"] = "error"
        record["secondary_scrape_error"] = clean_text(exc)

    return record


def run_secondary_scrape(input_csv_path):
    input_df = pd.read_csv(input_csv_path)
    if "listing_url" not in input_df.columns:
        raise ValueError("The input CSV must contain a listing_url column.")

    if MAX_LISTINGS is not None:
        input_df = input_df.head(MAX_LISTINGS).copy()

    unique_urls = (
        input_df["listing_url"]
        .dropna()
        .astype(str)
        .drop_duplicates()
        .tolist()
    )

    opener = build_http_opener()
    secondary_records = []

    for index, listing_url in enumerate(unique_urls, start=1):
        print(f"[{index}/{len(unique_urls)}] Fetching secondary details: {listing_url}")
        secondary_records.append(fetch_secondary_details(listing_url, opener))
        time.sleep(REQUEST_DELAY_SECONDS)

    secondary_df = pd.DataFrame(secondary_records).reindex(columns=SECONDARY_COLUMNS)
    secondary_df = secondary_df.where(pd.notna(secondary_df), None)

    merged_df = input_df.merge(secondary_df, on="listing_url", how="left")
    output_csv_path = build_output_csv_path(input_csv_path)
    merged_df.to_csv(output_csv_path, index=False, encoding="utf-8")

    return input_df, secondary_df, merged_df, output_csv_path

In [ ]:
input_csv_path = find_input_csv_path(INPUT_CSV_PATH)
output_csv_path = build_output_csv_path(input_csv_path)

print(f"Input CSV: {input_csv_path}")
print(f"Output CSV: {output_csv_path}")

input_df, secondary_df, merged_df, output_csv_path = run_secondary_scrape(input_csv_path)

print(f"Fetched secondary details for {len(secondary_df)} unique listings.")
print(f"Saved merged file to {output_csv_path}")

secondary_df.head()

Input CSV: scrape_audi_q4_20260507.csv
Output CSV: scrape_audi_q4_20260507_secondary.csv
[1/1339] Fetching secondary details: https://www.autoscout24.com/offers/audi-q4-e-tron-s-line-45-quattro-matrix-pano-acc-a-electric-blue-5e239b7a-4001-4ddd-a84c-447f37d94879
[2/1339] Fetching secondary details: https://www.autoscout24.com/offers/audi-q4-e-tron-40-s-line-electric-grey-f8da03cf-a310-496e-8958-f6d2d05d7514
[3/1339] Fetching secondary details: https://www.autoscout24.com/offers/audi-q4-e-tron-40-navi-plus-led-kamera-ahk-pdc-electric-grey-f423afed-a4c7-49bf-a1eb-b9c001c115cb
[4/1339] Fetching secondary details: https://www.autoscout24.com/offers/audi-q4-e-tron-electric-grey-f8f86c37-5a97-4f77-bbbb-37b04dc359cf
[5/1339] Fetching secondary details: https://www.autoscout24.com/offers/audi-q4-e-tron-sportback-e-tron-40-ahk-navi-rfk-waermepumpe-electric-black-16f816e4-61e0-4b37-a728-8fd1abc8eb38
[6/1339] Fetching secondary details: https://www.autoscout24.com/offers/audi-q4-e-tron-sportback-

,listing_url,warranty_exists,warranty_text,has_full_service_history,had_accident,damage_conditions,previous_owner_count,body_type,door_count,seat_count,exterior_color,paint_type,interior_color,upholstery_material,battery_ownership,battery_charging_time,electric_range,electric_range_city,secondary_scrape_status,secondary_scrape_error
0,https://www.autoscout24.com/offers/audi-q4-e-t...,True,None,True,False,None,1.0,SUV/Off-Road/Pick-Up,5.0,5.0,Blue,Metallic,Beige,Full leather,None,None,412 km,None,ok,None
1,https://www.autoscout24.com/offers/audi-q4-e-t...,True,12 months,True,False,None,1.0,SUV/Off-Road/Pick-Up,5.0,5.0,Grey,Metallic,Black,Cloth,None,None,None,None,ok,None
2,https://www.autoscout24.com/offers/audi-q4-e-t...,True,None,True,False,None,1.0,SUV/Off-Road/Pick-Up,4.0,5.0,Grey,None,Black,Cloth,None,None,396 km,None,ok,None
3,https://www.autoscout24.com/offers/audi-q4-e-t...,True,None,False,False,None,1.0,SUV/Off-Road/Pick-Up,5.0,5.0,Grey,Metallic,Black,None,None,None,441 km,None,ok,None
4,https://www.autoscout24.com/offers/audi-q4-e-t...,True,12 months,False,False,None,1.0,SUV/Off-Road/Pick-Up,5.0,5.0,Black,Metallic,Black,Cloth,None,None,527 km,None,ok,None
